
# Gas de Chaplygin generalizado: análisis dinámico y ajuste bayesiano

Replicación de **Szydłowski & Czaja**, *Stability of FRW cosmology with generalized Chaplygin gas*,
Phys. Rev. D **69**, 023506 (2004) — [arXiv:astro-ph/0306579].

Ajuste del modelo con:

- **Cronómetros cósmicos y BAO** → datos de $H(z)$
- **Supernovas tipo Ia (Pantheon)** → magnitudes aparentes $m_B(z)$, con matriz de covarianza
  completa (estadística + sistemática) y marginalización analítica sobre la magnitud absoluta.

---

**Modelo.** Para el universo FRW plano con materia y gas de Chaplygin generalizado
(ecuaciones 18 y 33 del artículo):

$$
E(z)^2 \equiv \frac{H(z)^2}{H_0^2}
= \Omega_m (1+z)^3
+ (1-\Omega_m)\Big[A_s + (1-A_s)(1+z)^{3(1+\alpha)}\Big]^{\frac{1}{1+\alpha}}
$$

con $0<\alpha\le 1$ (condición $c_s\le c$ del artículo) y $A_s = A/\rho_0^{1+\alpha}\in(0,1)$.
Por construcción $E(0)=1$.

*Prácticas preprofesionales 2025-II — GRACOCC, UNT*

## 1. Librerías

In [ ]:
import os
import zipfile

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid
from scipy.optimize import minimize
from scipy.stats import chi2 as chi2_dist

import emcee
import corner

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

C_LIGHT = 299792.458   # velocidad de la luz [km/s]
print("emcee", emcee.__version__)


## 2. Datos observacionales de $H(z)$

42 mediciones de la tasa de expansión obtenidas con cronómetros cósmicos y BAO.
Columnas: $z$, $H(z)$ en km s⁻¹ Mpc⁻¹, y su incertidumbre $\sigma_H$.

In [ ]:
HZ_DATA = np.array([
    [0.0708,  69.00, 19.68], [0.090,   69.00, 12.00], [0.070,  69.00, 19.68],
    [0.120,   68.60, 26.20], [0.170,   83.00,  8.00], [0.179,  75.00,  4.00],
    [0.199,   75.00,  5.00], [0.200,   72.90, 29.60], [0.240,  79.69,  2.65],
    [0.270,   77.00, 14.00], [0.280,   88.80, 36.60], [0.350,  84.40,  7.00],
    [0.352,   83.00, 14.00], [0.400,   95.00, 17.00], [0.430,  86.45,  3.68],
    [0.440,   82.60,  7.80], [0.480,   97.00, 62.00], [0.570,  92.40,  4.50],
    [0.593,  104.00, 13.00], [0.600,   87.90,  6.10], [0.680,  92.00,  8.00],
    [0.730,   97.30,  7.00], [0.781,  105.00, 12.00], [0.875, 125.00, 17.00],
    [0.880,   90.00, 40.00], [0.900,  117.00, 23.00], [1.037, 154.00, 20.00],
    [1.300,  168.00, 17.00], [1.363,  160.00, 33.60], [1.430, 177.00, 18.00],
    [1.530,  140.00, 14.00], [1.750,  202.00, 40.00], [1.965, 186.50, 50.40],
    [2.300,  224.00,  8.00], [2.340,  222.00,  7.00], [2.360, 226.00,  8.00],
    [0.934,  124.82,  1.51], [1.321,  149.70,  2.33], [1.484, 159.06,  6.40],
    [2.330,  233.79,  2.73], [0.922,  115.61,  1.39], [0.955, 114.70,  1.91],
])

_order = np.argsort(HZ_DATA[:, 0])
Z_HZ   = HZ_DATA[_order, 0]
H_OBS  = HZ_DATA[_order, 1]
H_ERR  = HZ_DATA[_order, 2]

print(f"{len(Z_HZ)} puntos, z = {Z_HZ.min():.3f} a {Z_HZ.max():.3f}")

plt.figure(figsize=(8, 5))
plt.errorbar(Z_HZ, H_OBS, yerr=H_ERR, fmt="o", ms=5, capsize=3,
             color="C0", ecolor="lightsteelblue")
plt.xlabel("$z$", fontsize=13)
plt.ylabel(r"$H(z)\ [\mathrm{km\,s^{-1}\,Mpc^{-1}}]$", fontsize=13)
plt.title("Datos observacionales de $H(z)$")
plt.grid(alpha=0.3)
plt.show()


## 3. El modelo cosmológico

Se definen la función de expansión $E(z)$, la tasa $H(z)$, la distancia luminosa

$$
d_C(z) = c\!\int_0^z \frac{dz'}{H(z')}, \qquad d_L(z) = (1+z)\,d_C(z),
$$

el módulo de distancia $\mu = 5\log_{10}(d_L/\mathrm{Mpc}) + 25$ y el parámetro
de desaceleración $q(z) = (1+z)E'/E - 1$.

La integral de $d_C$ se hace **una sola vez sobre una malla** y luego se interpola:
para las 1048 supernovas de Pantheon esto es órdenes de magnitud más rápido que
integrar por separado en cada $z_i$ dentro del MCMC.

In [ ]:
def E_model(z, alpha, As, Om):
    '''Función de expansión adimensional E(z) = H(z)/H0.

    Devuelve NaN donde el modelo no está definido (base negativa, alpha = -1),
    lo que la verosimilitud traduce después en -inf.
    '''
    z = np.asarray(z, dtype=float)
    if not np.isfinite(alpha) or np.isclose(1.0 + alpha, 0.0):
        return np.full(z.shape, np.nan)

    base = As + (1.0 - As) * (1.0 + z) ** (3.0 * (1.0 + alpha))
    with np.errstate(invalid="ignore", divide="ignore"):
        chapl = np.where(base > 0, base, np.nan) ** (1.0 / (1.0 + alpha))
        E2 = Om * (1.0 + z) ** 3 + (1.0 - Om) * chapl
        return np.sqrt(np.where(E2 > 0, E2, np.nan))


def H_model(z, alpha, As, H0, Om):
    '''H(z) en km/s/Mpc.'''
    return H0 * E_model(z, alpha, As, Om)


def luminosity_distance(z, alpha, As, H0, Om, n_grid=2048):
    '''Distancia luminosa d_L(z) en Mpc (modelo plano).'''
    z = np.atleast_1d(np.asarray(z, dtype=float))
    zg = np.linspace(0.0, float(np.max(z)), n_grid)

    Hg = H_model(zg, alpha, As, H0, Om)
    if not np.all(np.isfinite(Hg)) or np.any(Hg <= 0):
        return np.full(z.shape, np.nan)

    dC_grid = np.concatenate(([0.0], cumulative_trapezoid(C_LIGHT / Hg, zg)))
    return (1.0 + z) * np.interp(z, zg, dC_grid)


def distance_modulus(z, alpha, As, H0, Om):
    '''mu(z) = 5 log10(d_L / Mpc) + 25.'''
    dL = luminosity_distance(z, alpha, As, H0, Om)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(dL > 0, 5.0 * np.log10(dL) + 25.0, np.nan)


def deceleration_parameter(z, alpha, As, Om, dz=1e-4):
    '''q(z) = (1+z) E'(z)/E(z) - 1, por diferencias centradas.'''
    E  = E_model(z, alpha, As, Om)
    dE = (E_model(z + dz, alpha, As, Om) - E_model(z - dz, alpha, As, Om)) / (2 * dz)
    return (1.0 + z) * dE / E - 1.0


### 3.1 Verificaciones del modelo

Antes de ajustar conviene comprobar que el modelo hace lo que debe:

1. $E(0) = 1$ exactamente (normalización).
2. En el límite $A_s \to 1$ el gas se comporta como constante cosmológica, de modo
   que el modelo debe reducirse a $\Lambda$CDM.
3. En el límite $A_s \to 0$ el gas se comporta como polvo.

In [ ]:
# 1) Normalización
print("E(0) =", E_model(np.array([0.0]), 0.5, 0.75, 0.3)[0])

# 2) Límite LCDM (As -> 1)
z_test = np.array([0.5, 1.0, 2.0])
E_gcg  = E_model(z_test, 0.5, 1 - 1e-12, 0.3)
E_lcdm = np.sqrt(0.3 * (1 + z_test) ** 3 + 0.7)
print("As->1 :", np.max(np.abs(E_gcg - E_lcdm)), "(debe ser ~0)")

# 3) Límite de polvo (As -> 0)
E_dust_gcg = E_model(z_test, 0.5, 1e-12, 0.3)
E_dust     = np.sqrt((1 + z_test) ** 3)
print("As->0 :", np.max(np.abs(E_dust_gcg - E_dust)), "(debe ser ~0)")


## 4. Datos de Pantheon

**Punto importante.** La columna `mb` del archivo `lcparam_full_long.txt` es la magnitud
**aparente** $m_B$, no el módulo de distancia. La relación es $\mu = m_B - M$, donde $M$
es la magnitud absoluta de las SNe Ia, desconocida y degenerada con $H_0$.

Se trata $M$ como parámetro de estorbo y se **marginaliza analíticamente** con prior plano.
Definiendo $D_i = m_{B,i} - \mu_{\rm teo}(z_i)$ y con $C$ la covarianza total:

$$
A = D^{T} C^{-1} D, \qquad
B = D^{T} C^{-1} \mathbf{1}, \qquad
\mathcal{C} = \mathbf{1}^{T} C^{-1} \mathbf{1},
\qquad
\chi^2_{\rm marg} = A - \frac{B^2}{\mathcal{C}}
$$

Además se usa la covarianza **completa** $C = C_{\rm stat} + C_{\rm sys}$: quedarse solo con
la diagonal desperdicia la información sistemática, que es justamente lo que hace especial
a esta compilación.

In [ ]:
class PantheonData:
    '''Carga la compilación Pantheon (1048 SNe Ia) con su covarianza completa.'''

    def __init__(self, path):
        lc = self._find(path, "lcparam_full_long.txt")
        if lc is None:
            raise FileNotFoundError(
                f"No se encontró 'lcparam_full_long.txt' bajo '{path}'")

        raw = np.genfromtxt(lc, names=True, dtype=None, encoding="utf-8")
        self.zcmb = np.asarray(raw["zcmb"], dtype=float)
        self.zhel = np.asarray(raw["zhel"], dtype=float)
        self.mb   = np.asarray(raw["mb"],   dtype=float)
        self.dmb  = np.asarray(raw["dmb"],  dtype=float)
        self.n    = self.zcmb.size

        cov = self._load_sys(path)
        if cov is None:
            print("  [aviso] sin covarianza sistemática; se usa solo la estadística.")
            cov = np.zeros((self.n, self.n))
        cov = cov + np.diag(self.dmb ** 2)

        self.cinv = np.linalg.inv(cov)
        self.ones = np.ones(self.n)
        self.C_   = float(self.ones @ self.cinv @ self.ones)

    @staticmethod
    def _find(root, name):
        for dirpath, _, files in os.walk(root):
            if name in files:
                return os.path.join(dirpath, name)
        return None

    def _load_sys(self, path):
        txt = self._find(path, "sys_full_long.txt")
        if txt is not None:
            vals = np.loadtxt(txt)
        else:
            zp = self._find(path, "sys_full_long.zip")
            if zp is None:
                return None
            with zipfile.ZipFile(zp) as z:
                member = [m for m in z.namelist() if m.endswith(".txt")][0]
                vals = np.array([float(v) for v in
                                 z.read(member).decode("utf-8").split()])

        n = int(vals[0])                      # primer número: dimensión
        if n != self.n:
            raise ValueError(f"Covarianza {n}x{n} != {self.n} supernovas")
        return vals[1:].reshape((n, n))

    def chi2(self, alpha, As, H0, Om):
        '''chi^2 marginalizado analíticamente sobre la magnitud absoluta M.'''
        dL = luminosity_distance(self.zcmb, alpha, As, H0, Om)
        if not np.all(np.isfinite(dL)) or np.any(dL <= 0):
            return np.inf
        # corrección heliocéntrica: d_L = (1+z_hel) * d_C(z_cmb)
        dL = dL * (1.0 + self.zhel) / (1.0 + self.zcmb)

        D = self.mb - (5.0 * np.log10(dL) + 25.0)
        A = float(D @ self.cinv @ D)
        B = float(D @ self.cinv @ self.ones)
        return A - B * B / self.C_

    def best_M(self, alpha, As, H0, Om):
        '''Valor de M que minimiza el chi^2 (para graficar).'''
        dL = luminosity_distance(self.zcmb, alpha, As, H0, Om)
        dL = dL * (1.0 + self.zhel) / (1.0 + self.zcmb)
        D = self.mb - (5.0 * np.log10(dL) + 25.0)
        return float(D @ self.cinv @ self.ones) / self.C_


Indica abajo la carpeta donde descomprimiste `data_Pantheon.zip`. Si la dejas en `None`
(o si los archivos no están), el notebook sigue funcionando **solo con $H(z)$**.

In [ ]:
PANTHEON_PATH = None      # p. ej. "data_Pantheon"

pantheon = None
if PANTHEON_PATH:
    try:
        pantheon = PantheonData(PANTHEON_PATH)
        print(f"Pantheon cargado: {pantheon.n} supernovas, "
              f"z = {pantheon.zcmb.min():.4f} a {pantheon.zcmb.max():.4f}")
    except Exception as exc:
        print("No se pudo cargar Pantheon:", exc)
        print("Se continúa solo con H(z).")
else:
    print("PANTHEON_PATH = None -> ajuste solo con H(z).")

In [ ]:
# Diagrama de Hubble de los datos (solo si Pantheon está disponible)
if pantheon is not None:
    plt.figure(figsize=(9, 5.5))
    plt.errorbar(pantheon.zcmb, pantheon.mb, yerr=pantheon.dmb,
                 fmt="o", ms=2.5, alpha=0.5, color="C0", ecolor="lightsteelblue")
    plt.xscale("log")
    plt.xlabel("$z$", fontsize=13)
    plt.ylabel(r"$m_B$ (magnitud aparente)", fontsize=13)
    plt.title("Pantheon: magnitudes aparentes")
    plt.grid(alpha=0.3)
    plt.show()


## 5. Verosimilitud, priors y posterior

Los priors son planos pero con los rangos **físicos** del artículo:

| parámetro | rango | motivo |
|---|---|---|
| $\alpha$ | $(0,\,1]$ | garantiza $c_s^2 = \alpha A/\rho^{1+\alpha} \le 1$ |
| $A_s$ | $(0,\,1)$ | $A_s = A/\rho_0^{1+\alpha}$ es una fracción |
| $H_0$ | $(20,\,100)$ | rango amplio no informativo |
| $\Omega_m$ | $(0.01,\,0.6)$ | se deja **libre**, no fijo |

La log-verosimilitud es $\ln\mathcal{L} = -\chi^2/2$, sumando los conjuntos de datos activos.

In [ ]:
def chi2_hz(alpha, As, H0, Om):
    '''chi^2 de los datos de H(z).'''
    Hth = H_model(Z_HZ, alpha, As, H0, Om)
    if not np.all(np.isfinite(Hth)):
        return np.inf
    return float(np.sum(((H_OBS - Hth) / H_ERR) ** 2))


def chi2_total(alpha, As, H0, Om):
    '''chi^2 combinado de todos los conjuntos de datos activos.'''
    c = chi2_hz(alpha, As, H0, Om)
    if pantheon is not None and np.isfinite(c):
        c += pantheon.chi2(alpha, As, H0, Om)
    return c


def log_prior(theta):
    alpha, As, H0, Om = theta
    if not (0.0 < alpha <= 1.0):   return -np.inf
    if not (0.0 < As < 1.0):       return -np.inf
    if not (20.0 < H0 < 100.0):    return -np.inf
    if not (0.01 < Om < 0.6):      return -np.inf
    return 0.0


def log_posterior(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    c = chi2_total(*theta)
    if not np.isfinite(c):
        return -np.inf
    return lp - 0.5 * c


# prueba rápida
print("log_posterior en (0.5, 0.75, 70, 0.25) =",
      log_posterior([0.5, 0.75, 70.0, 0.25]))
print("log_posterior fuera del prior        =",
      log_posterior([2.0, 0.75, 70.0, 0.25]))


## 6. Muestreo MCMC con `emcee`

Los caminantes se inicializan en una bola pequeña, **verificando que cada uno arranque
con posterior finito** (si un caminante nace fuera del prior, la cadena se queda pegada).

El burn-in y el *thinning* no se fijan a ojo: se calculan a partir del **tiempo de
autocorrelación** $\tau$, que además sirve como diagnóstico de convergencia
(la regla usual es cadena $> 50\,\tau$).

In [ ]:
NDIM     = 4
NWALKERS = 48
NSTEPS   = 20000          # bájalo a ~4000 para una prueba rápida
LABELS   = [r"$\alpha$", r"$A_s$", r"$H_0$", r"$\Omega_m$"]
NAMES    = ["alpha", "A_s", "H0", "Om"]

p0_center = np.array([0.5, 0.75, 70.0, 0.25])
scale     = np.array([0.02, 0.02, 0.5, 0.01])

rng = np.random.default_rng(42)
p0 = []
while len(p0) < NWALKERS:
    cand = p0_center + scale * rng.standard_normal(NDIM)
    if np.isfinite(log_posterior(cand)):
        p0.append(cand)
p0 = np.array(p0)

sampler = emcee.EnsembleSampler(NWALKERS, NDIM, log_posterior)
sampler.run_mcmc(p0, NSTEPS, progress=True)

print("fracción de aceptación media =",
      np.round(np.mean(sampler.acceptance_fraction), 3))

In [ ]:
# Diagnóstico de convergencia y extracción de la cadena plana
tau = sampler.get_autocorr_time(tol=0)
tau_max = float(np.nanmax(tau))
print("tau =", np.round(tau, 1))
print(f"longitud de cadena = {NSTEPS/tau_max:.0f} x tau_max", 
      "  <-- OK" if NSTEPS > 50 * tau_max else "  <-- cadena corta, sube NSTEPS")

burn = int(min(5 * tau_max, 0.3 * NSTEPS))
thin = max(1, int(tau_max / 2))
flat = sampler.get_chain(discard=burn, thin=thin, flat=True)
print(f"burn-in = {burn}, thin = {thin}, muestras finales = {len(flat)}")

## 7. Resultados numéricos

In [ ]:
print(f"{'Parámetro':<10}{'mediana':>12}{'+68%':>10}{'-68%':>10}")
print("-" * 42)
res = {}
for i, name in enumerate(NAMES):
    q16, q50, q84 = np.percentile(flat[:, i], [16, 50, 84])
    res[name] = (q50, q84 - q50, q50 - q16)
    print(f"{name:<10}{q50:12.4f}{q84-q50:10.4f}{q50-q16:10.4f}")

best = [res[n][0] for n in NAMES]

chi2_b = chi2_total(*best)
ndof = len(Z_HZ) - NDIM
etiqueta = "H(z)"
if pantheon is not None:
    ndof += pantheon.n - 1        # -1 por la magnitud absoluta marginalizada
    etiqueta = "H(z) + Pantheon"

print(f"\nchi^2 ({etiqueta}) = {chi2_b:.2f}   g.l. = {ndof}   "
      f"chi^2/g.l. = {chi2_b/ndof:.3f}")

zq = np.linspace(0, 2, 500)
q  = deceleration_parameter(zq, best[0], best[1], best[3])
cruces = np.where(np.diff(np.sign(q)) != 0)[0]
print(f"q(0) = {q[0]:.3f}")
if cruces.size:
    print(f"transición desaceleración->aceleración en z_t = {zq[cruces[0]]:.3f}")

## 8. Figuras

### 8.1 Trazas de las cadenas

In [ ]:
chain = sampler.get_chain()
fig, axes = plt.subplots(NDIM, 1, figsize=(9, 8), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(chain[:, :, i], color="k", alpha=0.25, lw=0.4)
    ax.set_ylabel(LABELS[i])
axes[-1].set_xlabel("paso")
fig.suptitle("Trazas de las cadenas MCMC")
fig.tight_layout()
plt.show()

### 8.2 Distribuciones posteriores (corner plot)

In [ ]:
fig = corner.corner(flat, labels=LABELS, show_titles=True, title_fmt=".3f",
                    color="royalblue", quantiles=[0.16, 0.5, 0.84],
                    bins=40, smooth=1.0)
plt.show()


### 8.3 Ajuste de $H(z)$ con banda de credibilidad

La banda naranja contiene el 68% de las curvas $H(z)$ generadas por muestras del posterior:
dado el modelo y los datos, hay un 68% de probabilidad de que la curva verdadera esté ahí dentro.

In [ ]:
zg  = np.linspace(0, 2.6, 400)
idx = np.random.default_rng(0).choice(len(flat), min(400, len(flat)), replace=False)
draws = np.array([H_model(zg, *flat[i]) for i in idx])
lo, med, hi = np.percentile(draws, [16, 50, 84], axis=0)

plt.figure(figsize=(9, 6))
plt.errorbar(Z_HZ, H_OBS, yerr=H_ERR, fmt="o", ms=4, capsize=3,
             color="C0", ecolor="lightsteelblue", label="Datos $H(z)$")
plt.fill_between(zg, lo, hi, color="orange", alpha=0.35, label="Banda 68%")
plt.plot(zg, med, "r-", lw=2, label="Mediana del posterior")
plt.plot(zg, best[2] * np.sqrt(best[3] * (1 + zg) ** 3 + (1 - best[3])),
         "k--", lw=1.2, label=r"$\Lambda$CDM (mismo $\Omega_m$, $H_0$)")
plt.xlabel("$z$", fontsize=13)
plt.ylabel(r"$H(z)\ [\mathrm{km\,s^{-1}\,Mpc^{-1}}]$", fontsize=13)
plt.title("Gas de Chaplygin generalizado: ajuste de $H(z)$")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### 8.4 Regiones de confianza en el plano $(\alpha, A_s)$

Dos detalles metodológicos importantes:

1. En cada punto de la malla se **perfila** sobre $H_0$ y $\Omega_m$ (se minimiza el $\chi^2$
   respecto de ellos) en vez de fijarlos a un valor arbitrario. Fijarlos haría que los
   contornos dependieran de esa elección.
2. Los niveles son $\Delta\chi^2 = 2.30$ y $6.18$, correspondientes al 68.3% y 95.4% para
   **dos** parámetros de interés.

El eje $x$ es $\alpha$ y el eje $y$ es $A_s$ — conviene verificar siempre que las etiquetas
coincidan con el orden de los argumentos de `contour`.

In [ ]:
ag = np.linspace(0.01, 1.0, 45)
bg = np.linspace(0.55, 0.99, 45)
grid = np.zeros((len(bg), len(ag)))

def _nll(p, a, b):
    H0, Om = p
    if not (20 < H0 < 100 and 0.01 < Om < 0.6):
        return 1e6
    c = chi2_total(a, b, H0, Om)
    return c if np.isfinite(c) else 1e6

for i, b in enumerate(bg):
    for j, a in enumerate(ag):
        r = minimize(_nll, x0=[best[2], best[3]], args=(a, b),
                     method="Nelder-Mead",
                     options={"xatol": 1e-3, "fatol": 1e-3, "maxiter": 200})
        grid[i, j] = r.fun

chi2_min = grid.min()
d1 = chi2_dist.ppf(0.6827, 2)     # 2.30
d2 = chi2_dist.ppf(0.9545, 2)     # 6.18
print(f"chi2_min = {chi2_min:.2f}   niveles: +{d1:.2f}, +{d2:.2f}")

A, B = np.meshgrid(ag, bg)        # A -> eje x (alpha), B -> eje y (A_s)

plt.figure(figsize=(7.5, 6))
plt.contourf(A, B, grid, levels=[chi2_min, chi2_min + d1, chi2_min + d2],
             colors=["#2b5fa8", "#7fa8d8"], alpha=0.85)
cs = plt.contour(A, B, grid, levels=[chi2_min + d1, chi2_min + d2],
                 colors=["navy", "steelblue"], linewidths=1.2)
plt.clabel(cs, fmt={chi2_min + d1: "68.3%", chi2_min + d2: "95.4%"}, fontsize=9)

k = np.unravel_index(np.argmin(grid), grid.shape)
plt.plot(ag[k[1]], bg[k[0]], "r*", ms=15, label="mejor ajuste")
plt.xlabel(r"$\alpha$", fontsize=14)
plt.ylabel(r"$A_s$", fontsize=14)
plt.title(r"Regiones de confianza en $(\alpha, A_s)$")
plt.legend()
plt.show()


### 8.5 Parámetro de desaceleración

El cruce $q(z)=0$ marca la transición entre expansión desacelerada y acelerada.

In [ ]:
zq = np.linspace(0, 2.0, 300)
q_draws = np.array([deceleration_parameter(zq, flat[i][0], flat[i][1], flat[i][3])
                    for i in idx[:200]])
qlo, qmed, qhi = np.percentile(q_draws, [16, 50, 84], axis=0)

plt.figure(figsize=(8, 5.5))
plt.fill_between(zq, qlo, qhi, color="orange", alpha=0.35, label="68%")
plt.plot(zq, qmed, "b-", lw=2, label="$q_{\\rm GCG}(z)$")
plt.axhline(0, color="k", lw=0.8)
cruces = np.where(np.diff(np.sign(qmed)) != 0)[0]
if cruces.size:
    z_t = zq[cruces[0]]
    plt.axvline(z_t, color="green", ls=":", label=f"$z_t \\approx {z_t:.2f}$")
plt.xlabel("$z$", fontsize=13)
plt.ylabel("$q(z)$", fontsize=13)
plt.title("Parámetro de desaceleración")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### 8.6 Distancia luminosa y diagrama de Hubble

La segunda celda solo se ejecuta si Pantheon está cargado. Nota que los datos se grafican
como $m_B - M$, con $M$ el valor que minimiza el $\chi^2$: es el mismo $M$ que se
marginalizó analíticamente en el ajuste.

In [ ]:
zs = np.linspace(0.001, 3.0, 400)
dL = luminosity_distance(zs, *best)

plt.figure(figsize=(8, 5.5))
plt.plot(zs, dL, "b-", lw=2)
plt.xlabel("$z$", fontsize=13)
plt.ylabel(r"$d_L(z)\ [\mathrm{Mpc}]$", fontsize=13)
plt.title("Distancia luminosa del gas de Chaplygin generalizado")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
if pantheon is not None:
    M_best = pantheon.best_M(*best)
    print(f"magnitud absoluta ajustada: M = {M_best:.4f}")

    zs  = np.linspace(pantheon.zcmb.min(), pantheon.zcmb.max(), 400)
    mus = 5 * np.log10(luminosity_distance(zs, *best)) + 25

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True,
                                   gridspec_kw={"height_ratios": [3, 1]})
    ax1.errorbar(pantheon.zcmb, pantheon.mb - M_best, yerr=pantheon.dmb,
                 fmt="o", ms=2.5, alpha=0.5, color="C0",
                 ecolor="lightsteelblue", label="Pantheon")
    ax1.plot(zs, mus, "r-", lw=2, label="GCG (mejor ajuste)")
    ax1.set_ylabel(r"$\mu(z)$", fontsize=13)
    ax1.set_title("Diagrama de Hubble — Pantheon")
    ax1.legend(); ax1.grid(alpha=0.3)

    mu_sn = 5 * np.log10(luminosity_distance(pantheon.zcmb, *best)
                         * (1 + pantheon.zhel) / (1 + pantheon.zcmb)) + 25
    ax2.errorbar(pantheon.zcmb, pantheon.mb - M_best - mu_sn, yerr=pantheon.dmb,
                 fmt="o", ms=2.5, alpha=0.5, color="C0", ecolor="lightsteelblue")
    ax2.axhline(0, color="r", lw=1.5)
    ax2.set_xscale("log")
    ax2.set_xlabel("$z$", fontsize=13)
    ax2.set_ylabel("residuos")
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Pantheon no cargado: se omite el diagrama de Hubble.")


## 9. Comentarios sobre los resultados

Al ajustar **solo con $H(z)$** aparecen dos rasgos que conviene reportar tal cual:

- El posterior de $A_s$ se acumula contra el borde $A_s \to 1$ y el de $\alpha$ contra
  $\alpha \to 0$. Ese es precisamente el **límite $\Lambda$CDM** del gas de Chaplygin:
  los datos de $H(z)$ por sí solos no exigen $\alpha \neq 0$, es decir, no hay evidencia
  a favor del gas frente a la constante cosmológica.
- Hay una **degeneración fuerte entre $A_s$ y $\Omega_m$**, visible como una banda
  inclinada en el corner plot. Es esperable: el término $(1-A_s)(1+z)^{3(1+\alpha)}$ del
  gas se comporta como materia y compite con $\Omega_m$. Por eso el modelo original de
  Chaplygin se planteaba como *unificación* de materia y energía oscura, sin un
  $\Omega_m$ separado.

Dos variantes que vale la pena probar como extensión:

1. Añadir Pantheon (`PANTHEON_PATH`), que al restringir las distancias rompe parcialmente
   la degeneración anterior.
2. Ajustar el modelo **unificado**, con $\Omega_m = 0$, de modo que el gas dé cuenta de
   toda la materia oscura y la energía oscura a la vez; es la versión que motiva el paper.